# 曲线演化

## 可视化
* 因为曲线的特殊性（无法直接用netgen可视化），因此有如下两种可视化的办法：
    * 使用曲线网格，并导出vtk（需要自定义很多函数）
    * 使用domain网格，并限制到边界上生成曲线的有限元空间来进行演化计算

这里采用第二种方法，下面的结果是个示例

## 第二种方法的好处和坏处
* 好处，不用各种自定义的函数
* 坏处，计算过程无法可视化；有限元空间无法可视化（有些步骤需要仔细判断到底对不对）；计算结果都存储完之后进行可视化
* 为了方便，这里只用线性元，并用matplotlib来进行可视化

In [45]:
## Do not delete, Do not Run

## 背景

In [1]:
from ngsolve import *
import numpy as np
from netgen.geom2d import SplineGeometry

In [2]:
from scipy.sparse import *
import numpy as np
from esfem import MyInv, Pos_Transformer

### Setting of Velocity field

这是一个 `TwoTimeVelocityField` 类的实例。速度场的表达式如下：分为径向的速度和一个旋转的速度场。旋转的角速度会根据幅角不同，在正 $y$ 轴上最小，为 $0.2$，在负 $y$ 轴上最大为 $2.2$。

* Expression of $u$; Hessian $D^2u$ and Gradient $Du$

\begin{align}
u(x,t) &= x(1-|x|^2) + \left(1.2 - \frac{y}{|x|} \right)(-y,x)
\end{align}

<div style="text-align: center;">
<img src="../../data/results/Velocity.png" alt="Velocity" width="400"/>
</div>

In [18]:
def GetHessian(u,dim):
    '''
    Get the Hessian of a scalar field u.
    '''
    if dim == 2:
        hessian = CF((u.Diff(x).Diff(x).Compile(), u.Diff(x).Diff(y).Compile(),
                  u.Diff(y).Diff(x).Compile(), u.Diff(y).Diff(y).Compile()), 
                  dims=(2,2))
    elif dim == 3:
        hessian = CF((u.Diff(x).Diff(x).Compile(), u.Diff(x).Diff(y).Compile(), u.Diff(x).Diff(z).Compile(),
                  u.Diff(y).Diff(x).Compile(), u.Diff(y).Diff(y).Compile(), u.Diff(y).Diff(z).Compile(),
                  u.Diff(z).Diff(x).Compile(), u.Diff(z).Diff(y).Compile(), u.Diff(z).Diff(z).Compile()), 
                  dims=(3,3))
    return hessian

def GetGradient(u):
    '''
    Get the Gradient of a vector field u (CF function). (Grad u)_ij = D_i u_j
    '''
    dim = u.dim
    if dim == 2:
        gradient = CF((u[0].Diff(x).Compile(), u[1].Diff(x).Compile(),
                       u[0].Diff(y).Compile(), u[1].Diff(y).Compile(),), dims=2)
    elif dim == 3:
        gradient = CF((u[0].Diff(x).Compile(), u[1].Diff(x).Compile(), u[2].Diff(x).Compile(),
                       u[0].Diff(y).Compile(), u[1].Diff(y).Compile(), u[2].Diff(y).Compile(),
                       u[0].Diff(z).Compile(), u[1].Diff(z).Compile(), u[2].Diff(z).Compile(),), dims=3)
    return gradient

class TwoTimeVelocityField():
    '''
    A class to represent an autonomous velocity field with parameter mu.

    Attributes
    ----------
    mu : float
        A parameter for the velocity field.
    coef : float
        A coefficient used in the velocity field calculations.
    u : CF
        The velocity field as a function of x and y.
    mat_D2u : list of CF
        The Cartesian Hessian of each component of the velocity field.
    mat_Du : list of list of CF
        The Cartesian Gradient of the velocity field.
    mat_Du_CF : CF
        The Cartesian Gradient of the velocity field as a single CF object.

    Methods
    -------
    __init__(self, mu, coef)
        Initializes the velocity field and its derivatives.
    '''
    def __init__(self,mu,coef) -> None:
        '''
        mu : 角速度的平均值
        coef : 角速度的振幅
        '''
        r = sqrt(x**2+y**2)
        self.v_r = r*(1-r**2)
        self.v_t = mu - coef*y/r
        self.mu = mu
        self.coef = coef
        self.u = CF((self.v_r * x/r - y * self.v_t,
                    self.v_r * y/r + x * self.v_t))

        # derived Plane Hessian for each component of velocity
        self.mat_D2u = [
            GetHessian(self.u[0],2), GetHessian(self.u[1],2)
        ]
        # Plane Gradient -- Du: first row: D u_0, second row D u_1
        # (Du)_ij = D_j u_i，is the transpose of the gradient matrix
        self.mat_Du = [
            [self.u[0].Diff(x).Compile(), self.u[0].Diff(y).Compile()],
            [self.u[1].Diff(x).Compile(), self.u[1].Diff(y).Compile()]
        ]
        self.mat_Du_CF = GetGradient(self.u).trans

#### 几何设定以及速度场的解析表达式
* 初始的曲线是一个圆周，中心在 $x_0,y_0$，半径为 $r_0$
* 初始曲面的法向量场的表达式、梯度法向量场的lift以及投影算子
* nCF是精确的法向量，只是为了插值得到后面的初始时刻的法向量，可以近似由曲面的几何法向量 specialcf.normal 来替代
* DnCF是Weingarten矩阵的表达式，只是为了插值后文初始时刻Weingarten矩阵的初值sgnCF。可以用高阶曲面近似的Weingarten矩阵来替代

#### CF的字典
* 把CF function写进一个字典中然后调用，突出它们作为CF的特点

#### weingarten_map的数学表达：

* ...解析表达式计算的都是lift之后的表达式

In [6]:
class MoveCircleGeo():
    def __init__(self, center_x, center_y, radius) -> None:
        self.center_x = center_x
        self.center_y = center_y
        self.radius = radius
        radial_distance = sqrt((x - self.center_x)**2 + (y - self.center_y)**2)
        
        self.coefficient_functions = {
            "normal_vector": CF(((x - self.center_x) / radial_distance, 
                                 (y - self.center_y) / radial_distance)),
            "weingarten_map": CF((( (y - self.center_y)**2 / radial_distance**3, 
                                   -(x - self.center_x) * (y - self.center_y) / radial_distance**3),
                                  (-(x - self.center_x) * (y - self.center_y) / radial_distance**3, 
                                    (x - self.center_x)**2 / radial_distance**3)), dims=(2, 2)),
            "projection_matrix": CF(((1 - ((x - self.center_x) / radial_distance)**2, 
                                     -(x - self.center_x) * (y - self.center_y) / radial_distance**2), 
                                    (-(x - self.center_x) * (y - self.center_y) / radial_distance**2, 
                                      1 - ((y - self.center_y) / radial_distance)**2)), dims=(2, 2))
        }

    def geocf(self, name):
        return self.coefficient_functions.get(name, None)
        # nabla ^2 d, weingarten mapping on parallel surfaces
        self.weingarten_map = CF((( (y - self.center_y)**2 / radial_distance**3, 
                       -(x - self.center_x) * (y - self.center_y) / radial_distance**3),
                      (-(x - self.center_x) * (y - self.center_y) / radial_distance**3, 
                        (x - self.center_x)**2 / radial_distance**3)), dims=(2, 2))
        self.projection_matrix = CF(((1 - ((x - self.center_x) / radial_distance)**2, 
                         -(x - self.center_x) * (y - self.center_y) / radial_distance**2), 
                        (-(x - self.center_x) * (y - self.center_y) / radial_distance**2, 
                          1 - ((y - self.center_y) / radial_distance)**2)), dims=(2, 2))


In [7]:
def TensorProduct(u,v):
    assert(type(u)==list)
    assert(type(v)==list)
    return CF((u[0]*v[0],u[0]*v[1],u[1]*v[0],u[1]*v[1]),dims=(2,2))

In [8]:
BDF_order = 1
if BDF_order == 1:
    ext_BDF = [1]
    CBDF    = [1, 1] # additional - except the first item
elif BDF_order == 2:
    ext_BDF = [2, -1]
    CBDF    = [3/2, 2, -1/2] # additional - except the first item
elif BDF_order == 3:
    ext_BDF = [3,-3,1]
    CBDF    = [11/6, 3, -3/2, 1/3] # additional - except the first item

def GetExt(lst,BDF_order):
    res = ext_BDF[0]*lst[0]
    for ii in range(BDF_order):
        if ii >= 1:
            res = res + ext_BDF[ii]*lst[ii]
    return res

### 设定速度场，初始几何的解析表达式

In [9]:
Info_VF = TwoTimeVF(1.2,1)
x0,y0,r0 = 1/4,1/4,1/2
Info_GEO = MoveCircleGeo(x0,y0,r0)
u = Info_VF.u
mat_D2u = Info_VF.mat_D2u
mat_Du = Info_VF.mat_Du
nCF_lift = Info_GEO.nCF_lift
DnCF_lift = Info_GEO.DnCF_lift

## 设定参数部分

In [22]:
dim = 2
tau0 = 2**(-7)
order = 1
print('spatial order is {}'.format(order))
BDF_order = 1

spatial order is 1


In [33]:
tauval = tau0/2
T = 8
maxh = 0.05
tau = Parameter(tauval)

## 初始网格、有限元空间、弱形式
* 仅仅涉及到boundary的有限元空间；因为此时曲面是一维曲线，直接生成比较麻烦，故将曲线作为曲面的边界，在生成有限元空间的时候使用平面的网格，但限制定义域为mesh boundary。这样生成的有限元空间会有冗余的自由度，用Compress将其压缩
* 因为曲线在netgen中不适合visualization，因此本算例将结果保存之后最后进行可视化

In [34]:
geo = SplineGeometry()
geo.AddCircle((x0,y0),r0,bc="circle")
mymesh = Mesh(geo.GenerateMesh(maxh=maxh))
mymesh.Curve(order)
bnd_fes = Compress(H1(mymesh,definedon=mymesh.Boundaries(".*"),order=order))
bnd_fesV = Compress(VectorH1(mymesh,definedon=mymesh.Boundaries(".*"),order=order))

In [35]:
# Working fem space for symmetric Weingarten map A
MatrixL2 = FESpace( [bnd_fes,bnd_fes,bnd_fes] )
#         n  weingarten
fes_pq = bnd_fesV*MatrixL2
p, qxx, qxy, qyy = fes_pq.TrialFunction()
pt, qtxx, qtxy, qtyy = fes_pq.TestFunction()
q = CoefficientFunction( (qxx, qxy,
                        qxy, qyy), dims=(2,2))
qt = CoefficientFunction( (qtxx, qtxy,
                        qtxy, qtyy), dims=(2,2))
fes_saddle = bnd_fes*bnd_fesV
chi, v_trial = fes_saddle.TrialFunction()
chit, v_test = fes_saddle.TestFunction()
n_trail, n_test = bnd_fesV.TnT()

### Exact\_sgn

* sgnCF 是Weingarten映射的表达式 是 切空间投影x梯度n

In [36]:
Initial_N = GridFunction(bnd_fesV)
Initial_N.Set(nCF_lift, definedon=mymesh.Boundaries('.*')) ## 这要如何检验？
PCF = Info_GEO.PCF 
sgnCF = PCF*DnCF_lift
# sgn is mean curvature (1/r) times ...
Exact_sgn00_2 = GridFunction(bnd_fes)
Exact_sgn01_2 = GridFunction(bnd_fes)
Exact_sgn11_2 = GridFunction(bnd_fes)
Exact_sgn00_2.Set(sgnCF[0,0],definedon=mymesh.Boundaries('.*'))
Exact_sgn01_2.Set(sgnCF[0,1],definedon=mymesh.Boundaries('.*'))
Exact_sgn11_2.Set(sgnCF[1,1],definedon=mymesh.Boundaries('.*'))

### BDF离散的方程变量
* gfu 对应于空间 fes_pq中的有限元函数，是p，q_ij的联合，生成了后面所有的分量有限元函数
* p: 法向量，q: Weingarten matrix只是把这些分量有限元函数再组装成向量和矩阵便于后文的矩阵计算
* v\_chi 对应于空间 fes_saddle，是v和乘子的联合
* 由于p,q是演化方程，需要设定初值

In [37]:
# %%BDF histroy data
BDF_hist_vars = ['p','q','v','gfu','v_chi','q_xx','q_xy','q_yy']
for var in BDF_hist_vars:
    locals()[var+'_hist'] = []

for ii in range(BDF_order):
    # For BDF scheme, save order-1 historic terms, the first is the nearest one 
    jj = BDF_order-1-ii
    locals()['gfu_BDF_'+str(ii)] = GridFunction(fes_pq)
    locals()['v_chi_BDF_'+str(ii)] = GridFunction(fes_saddle)
    locals()['chi_BDF_'+str(ii)], locals()['v_BDF_'+str(ii)] = locals()['v_chi_BDF_'+str(ii)].components
    locals()['p_BDF_'+str(ii)], locals()['q_xx_BDF_'+str(ii)], locals()['q_xy_BDF_'+str(ii)], locals()['q_yy_BDF_'+str(ii)] = locals()['gfu_BDF_'+str(ii)].components
    locals()['q_BDF_'+str(ii)] = CF((locals()['q_xx_BDF_'+str(ii)], locals()['q_xy_BDF_'+str(ii)], 
                                locals()['q_xy_BDF_'+str(ii)], locals()['q_yy_BDF_'+str(ii)]),dims=(2,2))

    # Initialize 
    texact = jj*tauval
    locals()['p_BDF_'+str(ii)].vec.data = Initial_N.vec
    locals()['q_xx_BDF_'+str(ii)].vec.data = BaseVector(Exact_sgn00_2.vec.FV().NumPy())
    locals()['q_xy_BDF_'+str(ii)].vec.data = BaseVector(Exact_sgn01_2.vec.FV().NumPy())
    locals()['q_yy_BDF_'+str(ii)].vec.data = BaseVector(Exact_sgn11_2.vec.FV().NumPy())

    for var in BDF_hist_vars:
        locals()[var+'_hist'].append(locals()[var+'_BDF_'+str(ii)])  

n_proj = GridFunction(bnd_fesV)
p_ext_1 = GetExt([xx[0] for xx in p_hist],BDF_order=BDF_order)
p_ext_2 = GetExt([xx[1] for xx in p_hist],BDF_order=BDF_order)
v_ext_1 = GetExt([xx[0] for xx in v_hist],BDF_order=BDF_order)
v_ext_2 = GetExt([xx[1] for xx in v_hist],BDF_order=BDF_order)
p_ext   = GetExt(p_hist,BDF_order=BDF_order)
q_ext   = GetExt(q_hist,BDF_order=BDF_order)
v_ext   = GetExt(v_hist,BDF_order=BDF_order)

In [38]:
# %%Bilinear forms and Linear forms Settings -- BDF extensions and 
Lhs_pq = BilinearForm(fes_pq)
Rhs_pq = LinearForm(fes_pq)

mat_pv = TensorProduct([p_ext_1,p_ext_2],[v_ext_1-u[0],v_ext_2-u[1]])
mat_Projp = TensorProduct([p_ext_1,p_ext_2],[p_ext_1,p_ext_2])
Id = CF((1,0,0,1),dims=(2,2))
mat_Ptn = Id - mat_Projp
mat_DuCF = CF((mat_Du[0][0],mat_Du[0][1],mat_Du[1][0],mat_Du[1][1]),dims=(2,2))
Du = mat_DuCF*mat_Ptn

# of unknowns
gradq_t_ele = [di for di in grad(qxx).Trace()]+[di for di in grad(qxy).Trace()]*2\
                +[di for di in grad(qyy).Trace()]
gradq_tensor = CF(tuple(gradq_t_ele),dims=(2,2,2))

## BDF2 修改物质导数
Lhs_pq += CBDF[0]/tau*InnerProduct(p,pt)*ds \
            - InnerProduct(grad(p).Trace()*(v_ext-u),pt)*ds
for jj in range(BDF_order):
    Rhs_pq += CBDF[jj+1]/tau*InnerProduct(p_hist[jj],pt)*ds
Rhs_pq += - InnerProduct(Du.trans*p_ext,pt)*ds

Lhs_pq += CBDF[0]/tau*InnerProduct(q,qt)*ds \
            - InnerProduct(gradq_tensor*(v_ext-u),qt)*ds
for jj in range(BDF_order):
    if jj == 0:
        gradvext = ext_BDF[jj]*grad(v_hist[jj]).Trace()
    else:
        gradvext = gradvext + ext_BDF[jj]*grad(v_hist[jj]).Trace()

for jj in range(BDF_order):
    Rhs_pq += CBDF[jj+1]/tau*InnerProduct(q_hist[jj],qt)*ds

Rhs_pq += InnerProduct(mat_pv*q_ext**2,qt)*ds\
            - InnerProduct(q_ext*Du,qt)*ds\
            - InnerProduct(Du.trans*q_ext,qt)*ds\
            + InnerProduct(q_ext*mat_DuCF.trans*mat_Projp,qt)*ds\
            + InnerProduct(mat_DuCF*p_ext,p_ext)*InnerProduct(q_ext,qt)*ds\
            + InnerProduct(mat_Projp*gradvext*q_ext,qt)*ds 

Hessu = [mat_Ptn*mat_D2u[ii]*mat_Ptn for ii in range(dim)]
Rhs_pq += -(InnerProduct(Hessu[0],qt)*p_ext_1+InnerProduct(Hessu[1],qt)*p_ext_2)*ds

Lhs_n = BilinearForm(bnd_fesV)
Rhs_n = LinearForm(bnd_fesV)
Lhs_n += InnerProduct(n_trail,n_test)*ds\
    + InnerProduct(grad(n_trail).Trace(),grad(n_test).Trace())*ds
Rhs_n += InnerProduct(p_ext,n_test)*ds + InnerProduct(q_ext,grad(n_test).Trace())*ds

Lhs_v = BilinearForm(fes_saddle)
Rhs_v = LinearForm(fes_saddle)
Lhs_v += InnerProduct(grad(v_trial).Trace(),grad(v_test).Trace())*ds\
        + chi*InnerProduct(n_proj,v_test)*ds\
        + chit*InnerProduct(n_proj,v_trial)*ds
Rhs_v += chit*InnerProduct(n_proj,u)*ds

# %%Iteration, note starting time for BDF2
told = (BDF_order-1)*tauval
Disp = GridFunction(bnd_fesV)

In [39]:
IniX = GridFunction(bnd_fesV)
IniX.Set(CF((x,y)),definedon=mymesh.Boundaries('.*'))
Pos_BND = Pos_Transformer(IniX)

In [40]:
import matplotlib.pyplot as plt
from tqdm import tqdm

In [41]:
Res = []

In [42]:
for ii in tqdm(range(int(T/tauval))):
    tauval = tau.Get()
    told = (ii+1)*tauval
    Lhs_n.Assemble()
    Rhs_n.Assemble()
    n_proj.vec.data = BaseVector(MyInv(Lhs_n.mat, Rhs_n.vec))

    Rhs_v.Assemble()
    Lhs_v.Assemble()
    for ii in range(BDF_order):
        jj = BDF_order - 1 - ii
        if jj > 0:
            v_chi_hist[jj].vec.data = BaseVector(v_chi_hist[jj-1].vec.FV().NumPy())
        elif jj == 0:
            v_chi_hist[jj].vec.data = BaseVector(MyInv(Lhs_v.mat,Rhs_v.vec))
    # 更新 v_o_sol

    # 已知 v_o_sol,p_o_sol,q_o_mat,更新 p,q
    Lhs_pq.Assemble()
    Rhs_pq.Assemble()
    for ii in range(BDF_order):
        jj = BDF_order - 1 - ii
        if jj > 0:
            gfu_hist[jj].vec.data = BaseVector(gfu_hist[jj-1].vec.FV().NumPy())
        elif jj == 0:
            gfu_hist[jj].vec.data = BaseVector(MyInv(Lhs_pq.mat,Rhs_pq.vec))

    Disp.vec.data += BaseVector(tauval*v_hist[0].vec.FV().NumPy())
    mymesh.SetDeformation(Disp)
    
    dispnp = Pos_Transformer(Disp)
    Possss = Pos_BND+dispnp
    Res.append(Possss)

100%|█████████████████████████████████████████████████████████████████████████| 2048/2048 [00:39<00:00, 52.37it/s]


In [45]:
import matplotlib.animation as animation
from IPython.display import display, HTML
# 更新函数，用于每一帧动画
fig, ax = plt.subplots(figsize=(8, 8),dpi=80)
nn = 16
num_frames = int(2048/nn)  # 减少帧数
def animate(ii):
    ax.clear()
    cax = ax.plot(Res[nn*ii][:,0], Res[nn*ii][:,1], 'o')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_xlim([-1,1])
    ax.set_ylim([-1,1])
    return cax

ani = animation.FuncAnimation(fig, animate, frames=num_frames, interval=200, blit=True)
# 显示动画
plt.close(fig)
display(HTML(ani.to_jshtml()))